# S6E9 — Public Anchor + Original HiRGE

The recipe is fixed: **98% rank of a public ensemble + 2% rank of our original
HiRGE-Fingerprint**. Most of this prediction comes from the public ensemble;
this is not evidence that our DL outperforms it or caused an improvement.

Attribution: [Taeyang's public Lexsort ensemble](https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master),
with its upstream [jazivxt anchor](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline)
and [Yekenot RealMLP](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch).
The reused output already contains its author's hand-set boundary adjustments
and leaderboard-selected sources. We do not claim these rules are invariant or
that resolving ties mathematically guarantees AUC gains.

Our DL was fitted independently on common five-fold splits, with inner-fold
target encodings: **OOF AUC 0.945498** (old HiRGE 0.942010). The complete anchor
has no retrieved matching OOF, so **full-blend CV is unavailable**.

Default mode reproduces the submitted cached predictions. Optional fresh GPU
training below is a new experiment, not guaranteed to reproduce the same LB.


In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
def unique(filename):
    hits = list(INPUT.rglob(filename))
    assert len(hits) == 1, f'Expected one {filename}; found {hits}'
    return hits[0]

LIB = unique('train_hirge_fingerprint.py').parent
ANCHOR_PATH = unique('submission.csv')
TEST_PATH = unique('test.csv')
COMP = TEST_PATH.parent
DL_PATH = LIB / 'fingerprint__test__test_hirge_fingerprint.npy'
OOF_PATH = LIB / 'fingerprint__oof__oof_hirge_fingerprint.npy'
print('Inputs:', LIB, ANCHOR_PATH, COMP, sep='\n')


In [ ]:
EXPECTED_ANCHOR = '57e8ee944c1ac39c0483bb977f1f9c67af419a105ca00942b953569802530ada'
EXPECTED_DL = '933c6f8b4c23207f76faefcc07957f6d04cdf36b9db0b2d31c4e7d790b219366'
EXPECTED_SUBMISSION = 'a5ae1e6490806b4f276749bdc4aa60925775d0648d4df3f3e9e48b5898fc1d67'
assert hashlib.sha256(ANCHOR_PATH.read_bytes()).hexdigest() == EXPECTED_ANCHOR, 'Public kernel version changed'
assert hashlib.sha256(DL_PATH.read_bytes()).hexdigest() == EXPECTED_DL, 'DL prediction version changed'
test = pd.read_csv(TEST_PATH, usecols=['id'])
anchor_frame = pd.read_csv(ANCHOR_PATH, float_precision='round_trip')
assert anchor_frame.id.is_unique and set(anchor_frame.id) == set(test.id)
anchor = anchor_frame.set_index('id').loc[test.id, 'Will_Buy_EV'].to_numpy(float)
dl = np.load(DL_PATH, allow_pickle=False)
assert anchor.shape == dl.shape == (286571,)
assert np.isfinite(anchor).all() and np.isfinite(dl).all()

train = pd.read_csv(COMP / 'train.csv', usecols=['Will_Buy_EV'])
y = train.Will_Buy_EV.eq('Yes').to_numpy()
oof = np.load(OOF_PATH, allow_pickle=False)
assert oof.shape == y.shape and np.isfinite(oof).all()
print('Our original HiRGE-Fingerprint OOF:', roc_auc_score(y, oof))
print('Full blend OOF: unavailable; the public anchor is test-only.')


## Optional: fit our original architecture from scratch

In [ ]:
TRAIN_FINGERPRINT = False
if TRAIN_FINGERPRINT:
    fresh = WORK / 'fresh_hirge'
    subprocess.run([sys.executable, str(LIB / 'train_hirge_fingerprint.py'),
                    '--data-dir', str(COMP), '--output-dir', str(fresh),
                    '--folds', '0', '1', '2', '3', '4', '--epochs', '12',
                    '--patience', '3', '--device', 'cuda'], check=True)
    dl = np.load(fresh / 'test/test_hirge_fingerprint.npy')
    fresh_oof = np.load(fresh / 'oof/oof_hirge_fingerprint.npy')
    print('Fresh OOF:', roc_auc_score(y, fresh_oof))
    print('Fresh blend LB is not measured. GPU training is not bitwise deterministic.')


In [ ]:
rank = lambda values: (rankdata(values, method='average') - .5) / len(values)
prediction = .98 * rank(anchor) + .02 * rank(dl)
assert np.isfinite(prediction).all() and ((prediction >= 0) & (prediction <= 1)).all()
submission = pd.DataFrame({'id': test.id, 'Will_Buy_EV': prediction})
assert submission.id.equals(test.id)
destination = WORK / 'submission.csv'
submission.to_csv(destination, index=False, float_format='%.17g', lineterminator='\r\n')
actual_sha = hashlib.sha256(destination.read_bytes()).hexdigest()
if not TRAIN_FINGERPRINT:
    assert actual_sha == EXPECTED_SUBMISSION, 'Output differs from the measured submission'
    print('Exact measured submission reproduced: public LB 0.94649.')
else:
    print('Fresh training output; submit separately to measure its public LB.')
print('SHA256:', actual_sha)
np.save(WORK / 'test_public98_hirge2.npy', prediction)
(WORK / 'provenance.json').write_text(json.dumps({
    'anchor_sha256': EXPECTED_ANCHOR, 'cached_dl_sha256': EXPECTED_DL,
    'submission_sha256': actual_sha, 'weights': [.98, .02],
    'fresh_training': TRAIN_FINGERPRINT, 'full_blend_cv': None,
    'measured_lb': None if TRAIN_FINGERPRINT else .94649,
}, indent=2))
submission.head()
